In [8]:
import pandas as pd
import numpy as np


In [9]:
#Load the data
df = pd.read_csv(r"C:\Users\vr380\OneDrive\Documents\panda_tutorial\it_employees_raw_dataset.csv")
df

,Employee_ID,Subfield,Entry_Year,Exit_Year,Status,Exit_Reason,Min_Package_LPA,Avg_Package_LPA,Max_Package_LPA,Experience_Years,City,Education_Level,Work_Mode
0,EMP01271,Networking,01/18,NaN,1,NaN,1.39,2.50,7.45,6.0,Bangalore,Bachelor's,Remote
1,EMP00457,IT SUPPORT & HELPDESK,2017,NaN,Yes,NaN,11.69,16.21,36.01,7.0,Mumbai,Master's,Remote
2,EMP00542,IT Support & Helpdesk,01/20,2020-01-01,N,higher studies,5.93,8.98,15.37,4.0,Gurgaon,Master's,Hybrid
3,EMP00940,CLOUD COMPUTING,2020,NaN,Active,NaN,7.16,9.63,26.09,4.0,Mumbai,NaN,Hybrid
4,EMP00582,UI/UX DESIGN,2018,NaN,ACTIVE,NaN,5.09,0.58,20.25,6.0,mumbai,Master's,Onsite
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2258,EMP01145,ai/ml engineering,2023.0,NaN,Active,NaN,10.39,15.93,44.36,1.0,Noida,Bachelor's,Hybrid
2259,EMP00049,DevOps,2021,NaN,1,NaN,7.01,10.88,23.77,3.0,GURGAON,Bachelor's,Remote
2260,EMP00773,IT SUPPORT & HELPDESK,2023,NaN,Yes,NaN,5.88,9.98,29.29,1.0,Pune,PhD,Work From Home
2261,EMP01849,database administration,2018,NaN,active,NaN,16.17,22.90,41.71,6.0,delhi,Bachelor's,WFH


In [10]:
# shape of data

print("Shape is:", df.shape)

Shape is: (2263, 13)


In [11]:
#dropping fully blank row

df = df.dropna(how ="all")
print("After dropping blank row" , df.shape)

After dropping blank row (2255, 13)


In [12]:
# Removing duplicate 

count_dup= df.duplicated().sum()
print("Duplicated rows are :", count_dup)
df = df.drop_duplicates()

dup_id_count = df["Employee_ID"].duplicated().sum()
print("number of duplicate IDs :" , dup_id_count)
df= df.drop_duplicates(subset=["Employee_ID"],keep="first")
print("After removing duplicates :" , df.shape)

Duplicated rows are : 55
number of duplicate IDs : 0
After removing duplicates : (2200, 13)


In [13]:
text_cols = ["Subfield", "City", "Exit_Reason", "Education_Level", "Work_Mode"]
 
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.replace("_", " ").str.title()
    df[col] = df[col].replace("Nan", np.nan)   
# merge same-meaning Work_Mode values
df["Work_Mode"] = df["Work_Mode"].replace({
    "Wfh": "Remote",
    "Work From Home": "Remote"
})
 
print("\nSubfield :", df["Subfield"].dropna().unique())
print(" ")
print("Work_Mode :", df["Work_Mode"].dropna().unique())


Subfield : ['Networking' 'It Support & Helpdesk' 'Cloud Computing' 'Ui/Ux Design'
 'Devops' 'Qa & Testing' 'Data Science' 'Ai/Ml Engineering'
 'Business Analysis' 'Cybersecurity' 'Database Administration'
 'Software Development']
 
Work_Mode : ['Remote' 'Hybrid' 'Onsite']


In [14]:
status_map = {
    "Y": "Active", "1": "Active", "Yes": "Active", "Working": "Active",
    "N": "Exited", "0": "Exited", "No": "Exited", "Left": "Exited", "Resigned": "Exited"
}
df["Status"] = df["Status"].astype(str).str.strip().replace(status_map)
df["Status"] = df["Status"].str.title()
df["Status"] = df["Status"].replace("Nan", np.nan)
 
print("\nStatus value counts:\n", df["Status"].value_counts(dropna=False))


Status value counts:
 Status
Active    1546
Exited     544
NaN        110
Name: count, dtype: int64


In [15]:
#clean entry_year

df["Entry_Year"] = df["Entry_Year"].astype(str).str.extract(r"(\d{4})")
df["Entry_Year"] = pd.to_numeric(df["Entry_Year"] , errors = "coerce")


print("\n Missing Year after extraction:", df["Entry_Year"].isnull().sum())
df = df.dropna(subset=["Entry_Year"])
df["Entry_Year"] = df["Entry_Year"].astype(int)


 Missing Year after extraction: 344


In [16]:
#clean Exit Year
df["Exit_Year"] = df["Exit_Year"].astype(str).str.extract(r"(\d{4})")
df["Exit_Year"] = pd.to_numeric(df["Exit_Year"], errors="coerce")  # keep NaN, don't drop
 
# logical fix: Active employees should NOT have an Exit_Year
df.loc[df["Status"] == "Active", "Exit_Year"] = np.nan
 

In [17]:
# clean Exit Reason columns

df.loc[df["Status"] == "Active", "Exit_Reason"] = np.nan


# Exited employeewith missing reason

df.loc[(df["Status"] == "Exited") & (df["Exit_Reason"].isnull()), "Exit_Reason"] = "Not Specified"


In [18]:
ackage_cols = ["Min_Package_LPA", "Avg_Package_LPA", "Max_Package_LPA"]
 
# 8a. fill missing values using subfield-wise median
for col in package_cols:
    df[col] = df.groupby("Subfield")[col].transform(lambda x: x.fillna(x.median()))
 
# 8b. fix logically broken values (Avg should be between Min and Max)
bad_avg = (df["Avg_Package_LPA"] < df["Min_Package_LPA"]) | (df["Avg_Package_LPA"] > df["Max_Package_LPA"] * 1.5)
print("\nLogically broken Avg_Package rows:", bad_avg.sum())
df.loc[bad_avg, "Avg_Package_LPA"] = np.nan
df["Avg_Package_LPA"] = df.groupby("Subfield")["Avg_Package_LPA"].transform(lambda x: x.fillna(x.median()))
 
# 8c. drop rows still logically inconsistent after fixing (rare edge cases)
still_bad = df[(df["Min_Package_LPA"] > df["Avg_Package_LPA"]) | (df["Avg_Package_LPA"] > df["Max_Package_LPA"])]
print("Remaining inconsistent rows (dropping):", still_bad.shape[0])
df = df.drop(still_bad.index)

NameError: name 'package_cols' is not defined

In [19]:
#Clean Experience year

df["Experience_Years"] = df["Experience_Years"].fillna(2024 - df["Entry_Year"])
df["Experience_Years"] = df["Experience_Years"].clip(lower=0)
df["Experience_Years"] = df["Experience_Years"].astype(int)


In [20]:
# filling missing values

df["Education_Level"] = df["Education_Level"].fillna("Not Specified")
df["City"] = df["City"].fillna("Not Specified")
df["Work_Mode"] = df["Work_Mode"].fillna("Not Specified")
df["Subfield"] = df["Subfield"].fillna("Not Specified")



In [21]:
# Final datatypes

df["Entry_Year"] = df["Entry_Year"].astype(int)
df["Exit_Year"] = df["Exit_Year"].astype("Int64")



In [22]:
# Final check

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDescribe:\n", df.describe())

Shape: (1856, 13)

Missing values:
 Employee_ID            0
Subfield               0
Entry_Year             0
Exit_Year           1422
Status                98
Exit_Reason         1354
Min_Package_LPA        0
Avg_Package_LPA       98
Max_Package_LPA        0
Experience_Years       0
City                   0
Education_Level        0
Work_Mode              0
dtype: int64

Duplicate rows: 0

Describe:
         Entry_Year    Exit_Year  Min_Package_LPA  Avg_Package_LPA  \
count  1856.000000        434.0      1856.000000      1758.000000   
mean   2019.490302  2021.559908         6.066880         9.877056   
std       2.848731     2.350835         2.569582        12.296767   
min    2015.000000       2015.0         1.380000         0.202500   
25%    2017.000000       2020.0         4.190000         6.230000   
50%    2019.000000       2022.0         6.005000         9.120000   
75%    2022.000000       2024.0         7.780000        11.887500   
max    2024.000000       2024.0        16.4

In [23]:
df.to_csv("it_employees_cleaned.csv", index=False)
print("\nCleaned file saved as it_employees_cleaned.csv")
 


Cleaned file saved as it_employees_cleaned.csv


In [24]:
import os
print(os.getcwd())

C:\Users\vr380


In [26]:
import os

path = r'C:\Users\vr380\OneDrive\Documents\panda_tutorial\it_employees_cleaned.csv'

df.to_csv(path, index=False)
print("Saved at:", path)

Saved at: C:\Users\vr380\OneDrive\Documents\panda_tutorial\it_employees_cleaned.csv
